In [0]:
from pyspark.sql import SparkSession

spark=SparkSession.builder\
    .appName("Working with CSV and JSON")\
    .getOrCreate()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
doctors_df=spark.read.csv(
    "/Workspace/Users/azuser7210_mml.local@karthikirisoutlook.onmicrosoft.com/Drafts/doctors.csv",
    header=True,
    inferSchema=True
)

visits_df=spark.read.csv(
    "/Workspace/Users/azuser7210_mml.local@karthikirisoutlook.onmicrosoft.com/Drafts/visits.csv",
    header=True,
    inferSchema=True
)

hospital_df=spark.read.option(
    "multiline",
    "true"
).json(
    "/Workspace/Users/azuser7210_mml.local@karthikirisoutlook.onmicrosoft.com/Drafts/hospital_config.json"
)

In [0]:
doctors_df.show()

visits_df.show()

hospital_df.show(
    truncate=False
)

+---------+-----------+--------------+---------+----------------+----------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|
+---------+-----------+--------------+---------+----------------+----------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|            2000|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|            1000|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|              15|            2500|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|               6|            1200|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|            3000|
|     D107| Dr. Farhan|     Neurology|     Pune|               5|            1800|
|     D108|  Dr. Nisha|   Dermatology|    Kochi|               9|            1100|
+---------+-----------+--------------+---------+----------------+----------------+

+--

In [0]:
doctors_df.printSchema()

visits_df.printSchema()

hospital_df.printSchema()

root
 |-- doctor_id: string (nullable = true)
 |-- doctor_name: string (nullable = true)
 |-- specialization: string (nullable = true)
 |-- city: string (nullable = true)
 |-- experience_years: integer (nullable = true)
 |-- consultation_fee: integer (nullable = true)

root
 |-- visit_id: string (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- doctor_id: string (nullable = true)
 |-- visit_date: date (nullable = true)
 |-- diagnosis: string (nullable = true)
 |-- bill_amount: integer (nullable = true)
 |-- payment_status: string (nullable = true)

root
 |-- city: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- hospital_id: string (nullable = true)
 |-- hospital_name: string (nullable = true)
 |-- services: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [0]:
#1
doctors_df=spark.read.csv(
    "/Workspace/Users/azuser7210_mml.local@karthikirisoutlook.onmicrosoft.com/Drafts/doctors.csv",
    header=True,
    inferSchema=True
)

In [0]:
#2
visits_df=spark.read.csv(
    "/Workspace/Users/azuser7210_mml.local@karthikirisoutlook.onmicrosoft.com/Drafts/visits.csv",
    header=True,
    inferSchema=True
)

In [0]:
#3
doctors_df.printSchema()

visits_df.printSchema()

root
 |-- doctor_id: string (nullable = true)
 |-- doctor_name: string (nullable = true)
 |-- specialization: string (nullable = true)
 |-- city: string (nullable = true)
 |-- experience_years: integer (nullable = true)
 |-- consultation_fee: integer (nullable = true)

root
 |-- visit_id: string (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- doctor_id: string (nullable = true)
 |-- visit_date: date (nullable = true)
 |-- diagnosis: string (nullable = true)
 |-- bill_amount: integer (nullable = true)
 |-- payment_status: string (nullable = true)



In [0]:
#4
doctors_df.count()

8

In [0]:
#5
visits_df.count()

10

In [0]:
#6
doctors_df.filter(
    doctors_df.city==
    "Hyderabad"
).show()

+---------+-----------+--------------+---------+----------------+----------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|
+---------+-----------+--------------+---------+----------------+----------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|            3000|
+---------+-----------+--------------+---------+----------------+----------------+



In [0]:
#7
doctors_df.filter(
    doctors_df.specialization==
    "Cardiology"
).show()

+---------+-----------+--------------+---------+----------------+----------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|
+---------+-----------+--------------+---------+----------------+----------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|            3000|
+---------+-----------+--------------+---------+----------------+----------------+



In [0]:
#8
doctors_df.filter(
    doctors_df.experience_years>
    10
).show()

+---------+-----------+--------------+---------+----------------+----------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|
+---------+-----------+--------------+---------+----------------+----------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|              15|            2500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|            3000|
+---------+-----------+--------------+---------+----------------+----------------+



In [0]:
#9
visits_df.filter(
    visits_df.bill_amount>
    5000
).show()


+--------+------------+---------+----------+-------------+-----------+--------------+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_status|
+--------+------------+---------+----------+-------------+-----------+--------------+
|   V1004| Sneha Patel|     D104|2026-01-11|     Fracture|      12000|          Paid|
|   V1006|  Neha Singh|     D106|2026-01-12|Heart Checkup|       7000|          Paid|
|   V1009|   Kiran Rao|     D104|2026-01-14|    Back Pain|       6000|          Paid|
|   V1010| Nisha Reddy|     D101|2026-01-14|Heart Checkup|       5500|          Paid|
+--------+------------+---------+----------+-------------+-----------+--------------+



In [0]:
#10
visits_df.filter(
    visits_df.payment_status==
    "Pending"
).show()

+--------+------------+---------+----------+------------+-----------+--------------+
|visit_id|patient_name|doctor_id|visit_date|   diagnosis|bill_amount|payment_status|
+--------+------------+---------+----------+------------+-----------+--------------+
|   V1003|  Amit Kumar|     D103|2026-01-11|Skin Allergy|       2000|       Pending|
|   V1008|  Meera Nair|     D103|2026-01-13|Skin Allergy|       NULL|       Pending|
+--------+------------+---------+----------+------------+-----------+--------------+



In [0]:
#11
visits_df.filter(
    visits_df.payment_status==
    "Paid"
).show()

+--------+------------+---------+----------+-------------+-----------+--------------+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_status|
+--------+------------+---------+----------+-------------+-----------+--------------+
|   V1001|Rahul Sharma|     D101|2026-01-10|Heart Checkup|       5000|          Paid|
|   V1002| Priya Reddy|     D102|2026-01-10|     Migraine|       3500|          Paid|
|   V1004| Sneha Patel|     D104|2026-01-11|     Fracture|      12000|          Paid|
|   V1005|  Farhan Ali|     D105|2026-01-12|        Fever|       1500|          Paid|
|   V1006|  Neha Singh|     D106|2026-01-12|Heart Checkup|       7000|          Paid|
|   V1007| Arjun Verma|     D120|2026-01-13|     Migraine|       4000|          Paid|
|   V1009|   Kiran Rao|     D104|2026-01-14|    Back Pain|       6000|          Paid|
|   V1010| Nisha Reddy|     D101|2026-01-14|Heart Checkup|       5500|          Paid|
+--------+------------+---------+----------+----------

In [0]:
#12
doctors_df.groupBy(
    "specialization"
).avg(
    "consultation_fee"
).show()

+--------------+---------------------+
|specialization|avg(consultation_fee)|
+--------------+---------------------+
|   Orthopedics|               2500.0|
|    Cardiology|               2250.0|
|    Pediatrics|               1200.0|
|   Dermatology|               1050.0|
|     Neurology|               1900.0|
+--------------+---------------------+



In [0]:
#13
doctors_df.groupBy(
    "specialization"
).max(
    "consultation_fee"
).show()

+--------------+---------------------+
|specialization|max(consultation_fee)|
+--------------+---------------------+
|   Orthopedics|                 2500|
|    Cardiology|                 3000|
|    Pediatrics|                 1200|
|   Dermatology|                 1100|
|     Neurology|                 2000|
+--------------+---------------------+



In [0]:
#14
doctors_df.groupBy(
    "city"
).count().show()

+---------+-----+
|     city|count|
+---------+-----+
|    Delhi|    1|
|  Chennai|    1|
|    Kochi|    1|
|Hyderabad|    2|
|Bangalore|    1|
|     Pune|    1|
|   Mumbai|    1|
+---------+-----+



In [0]:
#15
doctors_df.groupBy(
    "specialization"
).count().show()

+--------------+-----+
|specialization|count|
+--------------+-----+
|   Orthopedics|    1|
|    Cardiology|    2|
|    Pediatrics|    1|
|   Dermatology|    2|
|     Neurology|    2|
+--------------+-----+



In [0]:
#16
from pyspark.sql.functions import *
from pyspark.sql.types import *

visits_df=visits_df.withColumn(
    "bill_amount",
    col("bill_amount").cast(
        "double"
    )
)

visits_df.select(
    sum(
        "bill_amount"
    ).alias(
        "total_bill_amount"
    )
).show()

+-----------------+
|total_bill_amount|
+-----------------+
|          46500.0|
+-----------------+



In [0]:
#17
visits_df.select(
    avg(
        "bill_amount"
    )
).show()

+-----------------+
| avg(bill_amount)|
+-----------------+
|5166.666666666667|
+-----------------+



In [0]:
#18
visits_df.select(
    max(
        "bill_amount"
    )
).show()

+----------------+
|max(bill_amount)|
+----------------+
|         12000.0|
+----------------+



In [0]:
#19
visits_df.select(
    min(
        "bill_amount"
    )
).show()

+----------------+
|min(bill_amount)|
+----------------+
|          1500.0|
+----------------+



In [0]:
#20
doctors_df.orderBy(
    col(
        "consultation_fee"
    ).desc()
).show()

+---------+-----------+--------------+---------+----------------+----------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|
+---------+-----------+--------------+---------+----------------+----------------+
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|            3000|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|              15|            2500|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|            2000|
|     D107| Dr. Farhan|     Neurology|     Pune|               5|            1800|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|               6|            1200|
|     D108|  Dr. Nisha|   Dermatology|    Kochi|               9|            1100|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|            1000|
+---------+-----------+--------------+---------+----------------+----------------+



In [0]:
#21
visits_df.orderBy(
    col(
        "bill_amount"
    ).desc()
).show()

+--------+------------+---------+----------+-------------+-----------+--------------+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_status|
+--------+------------+---------+----------+-------------+-----------+--------------+
|   V1004| Sneha Patel|     D104|2026-01-11|     Fracture|    12000.0|          Paid|
|   V1006|  Neha Singh|     D106|2026-01-12|Heart Checkup|     7000.0|          Paid|
|   V1009|   Kiran Rao|     D104|2026-01-14|    Back Pain|     6000.0|          Paid|
|   V1010| Nisha Reddy|     D101|2026-01-14|Heart Checkup|     5500.0|          Paid|
|   V1001|Rahul Sharma|     D101|2026-01-10|Heart Checkup|     5000.0|          Paid|
|   V1007| Arjun Verma|     D120|2026-01-13|     Migraine|     4000.0|          Paid|
|   V1002| Priya Reddy|     D102|2026-01-10|     Migraine|     3500.0|          Paid|
|   V1003|  Amit Kumar|     D103|2026-01-11| Skin Allergy|     2000.0|       Pending|
|   V1005|  Farhan Ali|     D105|2026-01-12|        Fe

In [0]:
#22
visits_df.filter(
    col(
        "bill_amount"
    ).isNull()
).show()

+--------+------------+---------+----------+------------+-----------+--------------+
|visit_id|patient_name|doctor_id|visit_date|   diagnosis|bill_amount|payment_status|
+--------+------------+---------+----------+------------+-----------+--------------+
|   V1008|  Meera Nair|     D103|2026-01-13|Skin Allergy|       NULL|       Pending|
+--------+------------+---------+----------+------------+-----------+--------------+



In [0]:
#23
visits_clean=visits_df.fillna({
    "bill_amount":0
})

visits_clean.show()

+--------+------------+---------+----------+-------------+-----------+--------------+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_status|
+--------+------------+---------+----------+-------------+-----------+--------------+
|   V1001|Rahul Sharma|     D101|2026-01-10|Heart Checkup|     5000.0|          Paid|
|   V1002| Priya Reddy|     D102|2026-01-10|     Migraine|     3500.0|          Paid|
|   V1003|  Amit Kumar|     D103|2026-01-11| Skin Allergy|     2000.0|       Pending|
|   V1004| Sneha Patel|     D104|2026-01-11|     Fracture|    12000.0|          Paid|
|   V1005|  Farhan Ali|     D105|2026-01-12|        Fever|     1500.0|          Paid|
|   V1006|  Neha Singh|     D106|2026-01-12|Heart Checkup|     7000.0|          Paid|
|   V1007| Arjun Verma|     D120|2026-01-13|     Migraine|     4000.0|          Paid|
|   V1008|  Meera Nair|     D103|2026-01-13| Skin Allergy|        0.0|       Pending|
|   V1009|   Kiran Rao|     D104|2026-01-14|    Back P

In [0]:
#24
visits_df.withColumn(
    "tax",
    col(
        "bill_amount"
    )*0.05
).show()

+--------+------------+---------+----------+-------------+-----------+--------------+-----+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_status|  tax|
+--------+------------+---------+----------+-------------+-----------+--------------+-----+
|   V1001|Rahul Sharma|     D101|2026-01-10|Heart Checkup|     5000.0|          Paid|250.0|
|   V1002| Priya Reddy|     D102|2026-01-10|     Migraine|     3500.0|          Paid|175.0|
|   V1003|  Amit Kumar|     D103|2026-01-11| Skin Allergy|     2000.0|       Pending|100.0|
|   V1004| Sneha Patel|     D104|2026-01-11|     Fracture|    12000.0|          Paid|600.0|
|   V1005|  Farhan Ali|     D105|2026-01-12|        Fever|     1500.0|          Paid| 75.0|
|   V1006|  Neha Singh|     D106|2026-01-12|Heart Checkup|     7000.0|          Paid|350.0|
|   V1007| Arjun Verma|     D120|2026-01-13|     Migraine|     4000.0|          Paid|200.0|
|   V1008|  Meera Nair|     D103|2026-01-13| Skin Allergy|       NULL|       Pen

In [0]:
#25
visits_df.withColumn(
    "tax",
    col(
        "bill_amount"
    )*0.05
).withColumn(
    "final_bill",
    col(
        "bill_amount"
    )+
    col(
        "tax"
    )
).show()

+--------+------------+---------+----------+-------------+-----------+--------------+-----+----------+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_status|  tax|final_bill|
+--------+------------+---------+----------+-------------+-----------+--------------+-----+----------+
|   V1001|Rahul Sharma|     D101|2026-01-10|Heart Checkup|     5000.0|          Paid|250.0|    5250.0|
|   V1002| Priya Reddy|     D102|2026-01-10|     Migraine|     3500.0|          Paid|175.0|    3675.0|
|   V1003|  Amit Kumar|     D103|2026-01-11| Skin Allergy|     2000.0|       Pending|100.0|    2100.0|
|   V1004| Sneha Patel|     D104|2026-01-11|     Fracture|    12000.0|          Paid|600.0|   12600.0|
|   V1005|  Farhan Ali|     D105|2026-01-12|        Fever|     1500.0|          Paid| 75.0|    1575.0|
|   V1006|  Neha Singh|     D106|2026-01-12|Heart Checkup|     7000.0|          Paid|350.0|    7350.0|
|   V1007| Arjun Verma|     D120|2026-01-13|     Migraine|     4000.0|   

In [0]:
#26
doctors_df.join(
    visits_df,
    "doctor_id",
    "inner"
).show()

+---------+-----------+--------------+---------+----------------+----------------+--------+------------+----------+-------------+-----------+--------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|visit_id|patient_name|visit_date|    diagnosis|bill_amount|payment_status|
+---------+-----------+--------------+---------+----------------+----------------+--------+------------+----------+-------------+-----------+--------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|   V1001|Rahul Sharma|2026-01-10|Heart Checkup|     5000.0|          Paid|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|            2000|   V1002| Priya Reddy|2026-01-10|     Migraine|     3500.0|          Paid|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|            1000|   V1003|  Amit Kumar|2026-01-11| Skin Allergy|     2000.0|       Pending|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|          

In [0]:
#27
doctors_df.join(
    visits_df,
    "doctor_id",
    "left"
).show()


+---------+-----------+--------------+---------+----------------+----------------+--------+------------+----------+-------------+-----------+--------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|visit_id|patient_name|visit_date|    diagnosis|bill_amount|payment_status|
+---------+-----------+--------------+---------+----------------+----------------+--------+------------+----------+-------------+-----------+--------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|   V1010| Nisha Reddy|2026-01-14|Heart Checkup|     5500.0|          Paid|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|            2000|   V1002| Priya Reddy|2026-01-10|     Migraine|     3500.0|          Paid|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|            1000|   V1008|  Meera Nair|2026-01-13| Skin Allergy|       NULL|       Pending|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|          

In [0]:
#27
doctors_df.join(
    visits_df,
    "doctor_id",
    "left"
).show()

+---------+-----------+--------------+---------+----------------+----------------+--------+------------+----------+-------------+-----------+--------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|visit_id|patient_name|visit_date|    diagnosis|bill_amount|payment_status|
+---------+-----------+--------------+---------+----------------+----------------+--------+------------+----------+-------------+-----------+--------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|   V1010| Nisha Reddy|2026-01-14|Heart Checkup|     5500.0|          Paid|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|            2000|   V1002| Priya Reddy|2026-01-10|     Migraine|     3500.0|          Paid|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|            1000|   V1008|  Meera Nair|2026-01-13| Skin Allergy|       NULL|       Pending|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|          

In [0]:
#29
doctors_df.join(
    visits_df,
    "doctor_id",
    "full"
).show()


+---------+-----------+--------------+---------+----------------+----------------+--------+------------+----------+-------------+-----------+--------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|visit_id|patient_name|visit_date|    diagnosis|bill_amount|payment_status|
+---------+-----------+--------------+---------+----------------+----------------+--------+------------+----------+-------------+-----------+--------------+
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|            2000|   V1002| Priya Reddy|2026-01-10|     Migraine|     3500.0|          Paid|
|     D120|       NULL|          NULL|     NULL|            NULL|            NULL|   V1007| Arjun Verma|2026-01-13|     Migraine|     4000.0|          Paid|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|            1000|   V1003|  Amit Kumar|2026-01-11| Skin Allergy|     2000.0|       Pending|
|     D103|  Dr. Anita|   Dermatology|  Chennai|          

In [0]:
#30
visits_df.join(
    doctors_df,
    "doctor_id",
    "left_anti"
).show()

+---------+--------+------------+----------+---------+-----------+--------------+
|doctor_id|visit_id|patient_name|visit_date|diagnosis|bill_amount|payment_status|
+---------+--------+------------+----------+---------+-----------+--------------+
|     D120|   V1007| Arjun Verma|2026-01-13| Migraine|     4000.0|          Paid|
+---------+--------+------------+----------+---------+-----------+--------------+



In [0]:
#31
doctors_df.join(
    visits_df,
    "doctor_id",
    "left_anti"
).show()

+---------+-----------+--------------+-----+----------------+----------------+
|doctor_id|doctor_name|specialization| city|experience_years|consultation_fee|
+---------+-----------+--------------+-----+----------------+----------------+
|     D107| Dr. Farhan|     Neurology| Pune|               5|            1800|
|     D108|  Dr. Nisha|   Dermatology|Kochi|               9|            1100|
+---------+-----------+--------------+-----+----------------+----------------+



In [0]:
#32
visits_df.groupBy(
    "doctor_id"
).count().show()


+---------+-----+
|doctor_id|count|
+---------+-----+
|     D102|    1|
|     D120|    1|
|     D103|    2|
|     D101|    2|
|     D105|    1|
|     D106|    1|
|     D104|    2|
+---------+-----+



In [0]:
#33
doctor_revenue=doctors_df.join(
    visits_df,
    "doctor_id",
    "inner"
).groupBy(
    "doctor_name"
).sum(
    "bill_amount"
)

doctor_revenue.show()

+-----------+----------------+
|doctor_name|sum(bill_amount)|
+-----------+----------------+
| Dr. Ramesh|         10500.0|
|  Dr. Meera|          1500.0|
| Dr. Suresh|         18000.0|
|  Dr. Anita|          2000.0|
|  Dr. Kiran|          7000.0|
|  Dr. Priya|          3500.0|
+-----------+----------------+



In [0]:
#34
doctors_df.join(
    visits_df,
    "doctor_id",
    "inner"
).groupBy(
    "doctor_name"
).sum(
    "bill_amount"
).orderBy(
    col(
        "sum(bill_amount)"
    ).desc()
).show(
    1
)


+-----------+----------------+
|doctor_name|sum(bill_amount)|
+-----------+----------------+
| Dr. Suresh|         18000.0|
+-----------+----------------+
only showing top 1 row


In [0]:

#35
doctors_df.join(
    visits_df,
    "doctor_id",
    "inner"
).groupBy(
    "specialization"
).sum(
    "bill_amount"
).orderBy(
    col(
        "sum(bill_amount)"
    ).desc()
).show(
    1
)


+--------------+----------------+
|specialization|sum(bill_amount)|
+--------------+----------------+
|   Orthopedics|         18000.0|
+--------------+----------------+
only showing top 1 row


In [0]:
#36
doctors_df.join(
    visits_df,
    "doctor_id",
    "inner"
).groupBy(
    "specialization"
).avg(
    "bill_amount"
).show()


+--------------+-----------------+
|specialization| avg(bill_amount)|
+--------------+-----------------+
|   Orthopedics|           9000.0|
|    Cardiology|5833.333333333333|
|    Pediatrics|           1500.0|
|   Dermatology|           2000.0|
|     Neurology|           3500.0|
+--------------+-----------------+



In [0]:
#37
doctors_df.join(
    visits_df,
    "doctor_id",
    "inner"
).groupBy(
    "city"
).sum(
    "bill_amount"
).show()


+---------+----------------+
|     city|sum(bill_amount)|
+---------+----------------+
|    Delhi|          1500.0|
|  Chennai|          2000.0|
|Hyderabad|         17500.0|
|Bangalore|          3500.0|
|   Mumbai|         18000.0|
+---------+----------------+



In [0]:
#38
doctors_df.join(
    visits_df,
    "doctor_id",
    "inner"
).groupBy(
    "doctor_name"
).count().show()

+-----------+-----+
|doctor_name|count|
+-----------+-----+
| Dr. Ramesh|    2|
|  Dr. Meera|    1|
| Dr. Suresh|    2|
|  Dr. Anita|    2|
|  Dr. Kiran|    1|
|  Dr. Priya|    1|
+-----------+-----+



In [0]:
#39
doctors_df.join(
    visits_df,
    "doctor_id",
    "inner"
).groupBy(
    "doctor_name"
).sum(
    "bill_amount"
).orderBy(
    col(
        "sum(bill_amount)"
    ).desc()
).show(
    3
)

+-----------+----------------+
|doctor_name|sum(bill_amount)|
+-----------+----------------+
| Dr. Suresh|         18000.0|
| Dr. Ramesh|         10500.0|
|  Dr. Kiran|          7000.0|
+-----------+----------------+
only showing top 3 rows


In [0]:
#40
doctors_df.join(
    visits_df,
    "doctor_id",
    "inner"
).groupBy(
    "doctor_name",
    "specialization",
    "city"
).agg(
    count(
        "visit_id"
    ).alias(
        "visit_count"
    ),
    sum(
        "bill_amount"
    ).alias(
        "total_revenue"
    ),
    avg(
        "bill_amount"
    ).alias(
        "average_revenue"
    )
).show()    

+-----------+--------------+---------+-----------+-------------+---------------+
|doctor_name|specialization|     city|visit_count|total_revenue|average_revenue|
+-----------+--------------+---------+-----------+-------------+---------------+
|  Dr. Anita|   Dermatology|  Chennai|          2|       2000.0|         2000.0|
|  Dr. Kiran|    Cardiology|Hyderabad|          1|       7000.0|         7000.0|
|  Dr. Meera|    Pediatrics|    Delhi|          1|       1500.0|         1500.0|
| Dr. Suresh|   Orthopedics|   Mumbai|          2|      18000.0|         9000.0|
| Dr. Ramesh|    Cardiology|Hyderabad|          2|      10500.0|         5250.0|
|  Dr. Priya|     Neurology|Bangalore|          1|       3500.0|         3500.0|
+-----------+--------------+---------+-----------+-------------+---------------+



In [0]:
#41
hospital_df=spark.read.option(
    "multiline",
    "true"
).json(
    "/Workspace/Users/azuser7210_mml.local@karthikirisoutlook.onmicrosoft.com/Drafts/hospital_config.json"
)

In [0]:
#42
hospital_df.printSchema()

root
 |-- city: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- hospital_id: string (nullable = true)
 |-- hospital_name: string (nullable = true)
 |-- services: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [0]:
#43
hospital_df.select(
    "hospital_name",
    col(
        "contact.phone"
    ).alias(
        "phone"
    )
).show()

+----------------+----------+
|   hospital_name|     phone|
+----------------+----------+
| Apollo Hospital|9876500011|
|Yashoda Hospital|      NULL|
|   Care Hospital|9876500013|
+----------------+----------+



In [0]:
#44
hospital_df.select(
    "hospital_name",
    col(
        "contact.email"
    ).alias(
        "email"
    )
).show()


+----------------+--------------------+
|   hospital_name|               email|
+----------------+--------------------+
| Apollo Hospital|[apollo@mail.com]...|
|Yashoda Hospital|[yashoda@mail.com...|
|   Care Hospital|                NULL|
+----------------+--------------------+



In [0]:
#45
hospital_df.filter(
    col(
        "contact.phone"
    ).isNull()
).show()

+---------+--------------------+-----------+----------------+--------------------+
|     city|             contact|hospital_id|   hospital_name|            services|
+---------+--------------------+-----------+----------------+--------------------+
|Hyderabad|{[yashoda@mail.co...|       H102|Yashoda Hospital|[Cardiology, Orth...|
+---------+--------------------+-----------+----------------+--------------------+



In [0]:
#46
hospital_df.filter(
    col(
        "contact.email"
    ).isNull()
).show()


+---------+------------------+-----------+-------------+--------------------+
|     city|           contact|hospital_id|hospital_name|            services|
+---------+------------------+-----------+-------------+--------------------+
|Bangalore|{NULL, 9876500013}|       H103|Care Hospital|[Neurology, Pedia...|
+---------+------------------+-----------+-------------+--------------------+



In [0]:
#47
hospital_df.withColumn(
    "phone",
    when(
        col(
            "contact.phone"
        ).isNull(),
        "Not Available"
    ).otherwise(
        col(
            "contact.phone"
        )
    )
).show()

+---------+--------------------+-----------+----------------+--------------------+-------------+
|     city|             contact|hospital_id|   hospital_name|            services|        phone|
+---------+--------------------+-----------+----------------+--------------------+-------------+
|Hyderabad|{[apollo@mail.com...|       H101| Apollo Hospital|[Cardiology, Neur...|   9876500011|
|Hyderabad|{[yashoda@mail.co...|       H102|Yashoda Hospital|[Cardiology, Orth...|Not Available|
|Bangalore|  {NULL, 9876500013}|       H103|   Care Hospital|[Neurology, Pedia...|   9876500013|
+---------+--------------------+-----------+----------------+--------------------+-------------+



In [0]:
#48
hospital_df.withColumn(
    "email",
    when(
        col(
            "contact.email"
        ).isNull(),
        "Not Available"
    ).otherwise(
        col(
            "contact.email"
        )
    )
).show()


+---------+--------------------+-----------+----------------+--------------------+--------------------+
|     city|             contact|hospital_id|   hospital_name|            services|               email|
+---------+--------------------+-----------+----------------+--------------------+--------------------+
|Hyderabad|{[apollo@mail.com...|       H101| Apollo Hospital|[Cardiology, Neur...|[apollo@mail.com]...|
|Hyderabad|{[yashoda@mail.co...|       H102|Yashoda Hospital|[Cardiology, Orth...|[yashoda@mail.com...|
|Bangalore|  {NULL, 9876500013}|       H103|   Care Hospital|[Neurology, Pedia...|       Not Available|
+---------+--------------------+-----------+----------------+--------------------+--------------------+



In [0]:
#49
hospital_df.select(
    "hospital_name",
    "city"
).show()

+----------------+---------+
|   hospital_name|     city|
+----------------+---------+
| Apollo Hospital|Hyderabad|
|Yashoda Hospital|Hyderabad|
|   Care Hospital|Bangalore|
+----------------+---------+



In [0]:
#50
hospital_df.select(
    "hospital_name",
    col(
        "contact.phone"
    ).alias(
        "phone"
    )
).show()

+----------------+----------+
|   hospital_name|     phone|
+----------------+----------+
| Apollo Hospital|9876500011|
|Yashoda Hospital|      NULL|
|   Care Hospital|9876500013|
+----------------+----------+



In [0]:
#51
hospital_df.withColumn(
    "service",
    explode(
        "services"
    )
).show()

+---------+--------------------+-----------+----------------+--------------------+-----------+
|     city|             contact|hospital_id|   hospital_name|            services|    service|
+---------+--------------------+-----------+----------------+--------------------+-----------+
|Hyderabad|{[apollo@mail.com...|       H101| Apollo Hospital|[Cardiology, Neur...| Cardiology|
|Hyderabad|{[apollo@mail.com...|       H101| Apollo Hospital|[Cardiology, Neur...|  Neurology|
|Hyderabad|{[apollo@mail.com...|       H101| Apollo Hospital|[Cardiology, Neur...|Dermatology|
|Hyderabad|{[yashoda@mail.co...|       H102|Yashoda Hospital|[Cardiology, Orth...| Cardiology|
|Hyderabad|{[yashoda@mail.co...|       H102|Yashoda Hospital|[Cardiology, Orth...|Orthopedics|
|Bangalore|  {NULL, 9876500013}|       H103|   Care Hospital|[Neurology, Pedia...|  Neurology|
|Bangalore|  {NULL, 9876500013}|       H103|   Care Hospital|[Neurology, Pedia...| Pediatrics|
+---------+--------------------+-----------+------

In [0]:
#51
hospital_df.withColumn(
    "service",
    explode(
        "services"
    )
).show()

+---------+--------------------+-----------+----------------+--------------------+-----------+
|     city|             contact|hospital_id|   hospital_name|            services|    service|
+---------+--------------------+-----------+----------------+--------------------+-----------+
|Hyderabad|{[apollo@mail.com...|       H101| Apollo Hospital|[Cardiology, Neur...| Cardiology|
|Hyderabad|{[apollo@mail.com...|       H101| Apollo Hospital|[Cardiology, Neur...|  Neurology|
|Hyderabad|{[apollo@mail.com...|       H101| Apollo Hospital|[Cardiology, Neur...|Dermatology|
|Hyderabad|{[yashoda@mail.co...|       H102|Yashoda Hospital|[Cardiology, Orth...| Cardiology|
|Hyderabad|{[yashoda@mail.co...|       H102|Yashoda Hospital|[Cardiology, Orth...|Orthopedics|
|Bangalore|  {NULL, 9876500013}|       H103|   Care Hospital|[Neurology, Pedia...|  Neurology|
|Bangalore|  {NULL, 9876500013}|       H103|   Care Hospital|[Neurology, Pedia...| Pediatrics|
+---------+--------------------+-----------+------

In [0]:
#53
hospital_df.groupBy(
    "city"
).count().show()


+---------+-----+
|     city|count|
+---------+-----+
|Hyderabad|    2|
|Bangalore|    1|
+---------+-----+



In [0]:
#54
hospital_df.withColumn(
    "service",
    explode(
        "services"
    )
).groupBy(
    "service"
).count().show()

+-----------+-----+
|    service|count|
+-----------+-----+
|Orthopedics|    1|
| Cardiology|    2|
| Pediatrics|    1|
|Dermatology|    1|
|  Neurology|    2|
+-----------+-----+



In [0]:
#55
hospital_df.filter(
    array_contains(
        col(
            "services"
        ),
        "Cardiology"
    )
).show()

+---------+--------------------+-----------+----------------+--------------------+
|     city|             contact|hospital_id|   hospital_name|            services|
+---------+--------------------+-----------+----------------+--------------------+
|Hyderabad|{[apollo@mail.com...|       H101| Apollo Hospital|[Cardiology, Neur...|
|Hyderabad|{[yashoda@mail.co...|       H102|Yashoda Hospital|[Cardiology, Orth...|
+---------+--------------------+-----------+----------------+--------------------+



In [0]:
#56
hospital_df.filter(
    array_contains(
        col(
            "services"
        ),
        "Neurology"
    )
).show()

+---------+--------------------+-----------+---------------+--------------------+
|     city|             contact|hospital_id|  hospital_name|            services|
+---------+--------------------+-----------+---------------+--------------------+
|Hyderabad|{[apollo@mail.com...|       H101|Apollo Hospital|[Cardiology, Neur...|
|Bangalore|  {NULL, 9876500013}|       H103|  Care Hospital|[Neurology, Pedia...|
+---------+--------------------+-----------+---------------+--------------------+



In [0]:
#57
hospital_df.filter(
    array_contains(
        col(
            "services"
        ),
        "Orthopedics"
    )
).show()


+---------+--------------------+-----------+----------------+--------------------+
|     city|             contact|hospital_id|   hospital_name|            services|
+---------+--------------------+-----------+----------------+--------------------+
|Hyderabad|{[yashoda@mail.co...|       H102|Yashoda Hospital|[Cardiology, Orth...|
+---------+--------------------+-----------+----------------+--------------------+



In [0]:
#58
hospital_df.filter(
    array_contains(
        col(
            "services"
        ),
        "Pediatrics"
    )
).show()

+---------+------------------+-----------+-------------+--------------------+
|     city|           contact|hospital_id|hospital_name|            services|
+---------+------------------+-----------+-------------+--------------------+
|Bangalore|{NULL, 9876500013}|       H103|Care Hospital|[Neurology, Pedia...|
+---------+------------------+-----------+-------------+--------------------+



In [0]:
#59
hospital_df.withColumn(
    "phone",
    col(
        "contact.phone"
    )
).withColumn(
    "email",
    col(
        "contact.email"
    )
).drop(
    "contact"
).write.mode(
    "overwrite"
).saveAsTable(
    "hospital_flattened"
)

In [0]:
#60
hospital_df.withColumn(
    "phone",
    col(
        "contact.phone"
    )
).withColumn(
    "email",
    col(
        "contact.email"
    )
).withColumn(
    "services_str",
    concat_ws(",", col("services"))
).drop(
    "contact",
    "services"
).write.mode(
    "overwrite"
).saveAsTable(
    "hospital_flattened_csv"
)

In [0]:
#61
from pyspark.sql.window import Window

doctor_revenue=doctors_df.join(
    visits_df,
    "doctor_id",
    "inner"
).groupBy(
    "doctor_id",
    "doctor_name",
    "specialization",
    "city"
).agg(
    sum(
        "bill_amount"
    ).alias(
        "revenue"
    )
)

window_spec=Window.orderBy(
    col(
        "revenue"
    ).desc()
)

doctor_revenue.withColumn(
    "rank",
    rank().over(
        window_spec
    )
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+----+
|doctor_id|doctor_name|specialization|     city|revenue|rank|
+---------+-----------+--------------+---------+-------+----+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|18000.0|   1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|10500.0|   2|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad| 7000.0|   3|
|     D102|  Dr. Priya|     Neurology|Bangalore| 3500.0|   4|
|     D103|  Dr. Anita|   Dermatology|  Chennai| 2000.0|   5|
|     D105|  Dr. Meera|    Pediatrics|    Delhi| 1500.0|   6|
+---------+-----------+--------------+---------+-------+----+



In [0]:
#62
doctor_revenue.withColumn(
    "dense_rank",
    dense_rank().over(
        window_spec
    )
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+----------+
|doctor_id|doctor_name|specialization|     city|revenue|dense_rank|
+---------+-----------+--------------+---------+-------+----------+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|18000.0|         1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|10500.0|         2|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad| 7000.0|         3|
|     D102|  Dr. Priya|     Neurology|Bangalore| 3500.0|         4|
|     D103|  Dr. Anita|   Dermatology|  Chennai| 2000.0|         5|
|     D105|  Dr. Meera|    Pediatrics|    Delhi| 1500.0|         6|
+---------+-----------+--------------+---------+-------+----------+



In [0]:
#63
doctor_revenue.withColumn(
    "row_number",
    row_number().over(
        window_spec
    )
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+----------+
|doctor_id|doctor_name|specialization|     city|revenue|row_number|
+---------+-----------+--------------+---------+-------+----------+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|18000.0|         1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|10500.0|         2|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad| 7000.0|         3|
|     D102|  Dr. Priya|     Neurology|Bangalore| 3500.0|         4|
|     D103|  Dr. Anita|   Dermatology|  Chennai| 2000.0|         5|
|     D105|  Dr. Meera|    Pediatrics|    Delhi| 1500.0|         6|
+---------+-----------+--------------+---------+-------+----------+



In [0]:
#64
doctor_revenue.orderBy(
    col(
        "revenue"
    ).desc()
).show(
    1
)

+---------+-----------+--------------+------+-------+
|doctor_id|doctor_name|specialization|  city|revenue|
+---------+-----------+--------------+------+-------+
|     D104| Dr. Suresh|   Orthopedics|Mumbai|18000.0|
+---------+-----------+--------------+------+-------+
only showing top 1 row


In [0]:
#65
doctor_revenue.orderBy(
    col(
        "revenue"
    ).desc()
).show(
    3
)

+---------+-----------+--------------+---------+-------+
|doctor_id|doctor_name|specialization|     city|revenue|
+---------+-----------+--------------+---------+-------+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|18000.0|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|10500.0|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad| 7000.0|
+---------+-----------+--------------+---------+-------+
only showing top 3 rows


In [0]:
#66
specialization_window=Window.partitionBy(
    "specialization"
).orderBy(
    col(
        "revenue"
    ).desc()
)

doctor_revenue.withColumn(
    "rank",
    rank().over(
        specialization_window
    )
).filter(
    col(
        "rank"
    )==1
).show()


+---------+-----------+--------------+---------+-------+----+
|doctor_id|doctor_name|specialization|     city|revenue|rank|
+---------+-----------+--------------+---------+-------+----+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|10500.0|   1|
|     D103|  Dr. Anita|   Dermatology|  Chennai| 2000.0|   1|
|     D102|  Dr. Priya|     Neurology|Bangalore| 3500.0|   1|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|18000.0|   1|
|     D105|  Dr. Meera|    Pediatrics|    Delhi| 1500.0|   1|
+---------+-----------+--------------+---------+-------+----+



In [0]:
#67
doctor_revenue.withColumn(
    "rank",
    rank().over(
        specialization_window
    )
).filter(
    col(
        "rank"
    )<=2
).show()

+---------+-----------+--------------+---------+-------+----+
|doctor_id|doctor_name|specialization|     city|revenue|rank|
+---------+-----------+--------------+---------+-------+----+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|10500.0|   1|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad| 7000.0|   2|
|     D103|  Dr. Anita|   Dermatology|  Chennai| 2000.0|   1|
|     D102|  Dr. Priya|     Neurology|Bangalore| 3500.0|   1|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|18000.0|   1|
|     D105|  Dr. Meera|    Pediatrics|    Delhi| 1500.0|   1|
+---------+-----------+--------------+---------+-------+----+



In [0]:
#68
doctor_revenue.withColumn(
    "running_revenue",
    sum(
        "revenue"
    ).over(
        window_spec
    )
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+---------------+
|doctor_id|doctor_name|specialization|     city|revenue|running_revenue|
+---------+-----------+--------------+---------+-------+---------------+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|18000.0|        18000.0|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|10500.0|        28500.0|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad| 7000.0|        35500.0|
|     D102|  Dr. Priya|     Neurology|Bangalore| 3500.0|        39000.0|
|     D103|  Dr. Anita|   Dermatology|  Chennai| 2000.0|        41000.0|
|     D105|  Dr. Meera|    Pediatrics|    Delhi| 1500.0|        42500.0|
+---------+-----------+--------------+---------+-------+---------------+



In [0]:
#69
doctor_revenue.withColumn(
    "previous_revenue",
    lag(
        "revenue"
    ).over(
        window_spec
    )
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+----------------+
|doctor_id|doctor_name|specialization|     city|revenue|previous_revenue|
+---------+-----------+--------------+---------+-------+----------------+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|18000.0|            NULL|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|10500.0|         18000.0|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad| 7000.0|         10500.0|
|     D102|  Dr. Priya|     Neurology|Bangalore| 3500.0|          7000.0|
|     D103|  Dr. Anita|   Dermatology|  Chennai| 2000.0|          3500.0|
|     D105|  Dr. Meera|    Pediatrics|    Delhi| 1500.0|          2000.0|
+---------+-----------+--------------+---------+-------+----------------+



In [0]:
#70
doctor_revenue.withColumn(
    "next_revenue",
    lead(
        "revenue"
    ).over(
        window_spec
    )
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+------------+
|doctor_id|doctor_name|specialization|     city|revenue|next_revenue|
+---------+-----------+--------------+---------+-------+------------+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|18000.0|     10500.0|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|10500.0|      7000.0|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad| 7000.0|      3500.0|
|     D102|  Dr. Priya|     Neurology|Bangalore| 3500.0|      2000.0|
|     D103|  Dr. Anita|   Dermatology|  Chennai| 2000.0|      1500.0|
|     D105|  Dr. Meera|    Pediatrics|    Delhi| 1500.0|        NULL|
+---------+-----------+--------------+---------+-------+------------+



In [0]:
#71
doctor_revenue.withColumn(
    "previous_revenue",
    lag(
        "revenue"
    ).over(
        window_spec
    )
).withColumn(
    "difference",
    col(
        "revenue"
    )-
    col(
        "previous_revenue"
    )
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+----------------+----------+
|doctor_id|doctor_name|specialization|     city|revenue|previous_revenue|difference|
+---------+-----------+--------------+---------+-------+----------------+----------+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|18000.0|            NULL|      NULL|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|10500.0|         18000.0|   -7500.0|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad| 7000.0|         10500.0|   -3500.0|
|     D102|  Dr. Priya|     Neurology|Bangalore| 3500.0|          7000.0|   -3500.0|
|     D103|  Dr. Anita|   Dermatology|  Chennai| 2000.0|          3500.0|   -1500.0|
|     D105|  Dr. Meera|    Pediatrics|    Delhi| 1500.0|          2000.0|    -500.0|
+---------+-----------+--------------+---------+-------+----------------+----------+



In [0]:
#72
doctor_revenue.withColumn(
    "next_revenue",
    lead(
        "revenue"
    ).over(
        window_spec
    )
).withColumn(
    "difference",
    col(
        "next_revenue"
    )-
    col(
        "revenue"
    )
).show()


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+------------+----------+
|doctor_id|doctor_name|specialization|     city|revenue|next_revenue|difference|
+---------+-----------+--------------+---------+-------+------------+----------+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|18000.0|     10500.0|   -7500.0|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|10500.0|      7000.0|   -3500.0|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad| 7000.0|      3500.0|   -3500.0|
|     D102|  Dr. Priya|     Neurology|Bangalore| 3500.0|      2000.0|   -1500.0|
|     D103|  Dr. Anita|   Dermatology|  Chennai| 2000.0|      1500.0|    -500.0|
|     D105|  Dr. Meera|    Pediatrics|    Delhi| 1500.0|        NULL|      NULL|
+---------+-----------+--------------+---------+-------+------------+----------+



In [0]:
#73
city_window=Window.partitionBy(
    "city"
).orderBy(
    col(
        "revenue"
    ).desc()
)

doctor_revenue.withColumn(
    "rank",
    rank().over(
        city_window
    )
).filter(
    col(
        "rank"
    )==1
).show()


+---------+-----------+--------------+---------+-------+----+
|doctor_id|doctor_name|specialization|     city|revenue|rank|
+---------+-----------+--------------+---------+-------+----+
|     D102|  Dr. Priya|     Neurology|Bangalore| 3500.0|   1|
|     D103|  Dr. Anita|   Dermatology|  Chennai| 2000.0|   1|
|     D105|  Dr. Meera|    Pediatrics|    Delhi| 1500.0|   1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|10500.0|   1|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|18000.0|   1|
+---------+-----------+--------------+---------+-------+----+



In [0]:
#74
city_window_low=Window.partitionBy(
    "city"
).orderBy(
    col(
        "revenue"
    ).asc()
)

doctor_revenue.withColumn(
    "rank",
    rank().over(
        city_window_low
    )
).filter(
    col(
        "rank"
    )==1
).show()


+---------+-----------+--------------+---------+-------+----+
|doctor_id|doctor_name|specialization|     city|revenue|rank|
+---------+-----------+--------------+---------+-------+----+
|     D102|  Dr. Priya|     Neurology|Bangalore| 3500.0|   1|
|     D103|  Dr. Anita|   Dermatology|  Chennai| 2000.0|   1|
|     D105|  Dr. Meera|    Pediatrics|    Delhi| 1500.0|   1|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad| 7000.0|   1|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|18000.0|   1|
+---------+-----------+--------------+---------+-------+----+



In [0]:
#75
doctor_revenue.withColumn(
    "rank",
    rank().over(
        window_spec
    )
).orderBy(
    "rank"
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+----+
|doctor_id|doctor_name|specialization|     city|revenue|rank|
+---------+-----------+--------------+---------+-------+----+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|18000.0|   1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|10500.0|   2|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad| 7000.0|   3|
|     D102|  Dr. Priya|     Neurology|Bangalore| 3500.0|   4|
|     D103|  Dr. Anita|   Dermatology|  Chennai| 2000.0|   5|
|     D105|  Dr. Meera|    Pediatrics|    Delhi| 1500.0|   6|
+---------+-----------+--------------+---------+-------+----+



In [0]:
#76
doctors_df.createOrReplaceTempView(
    "doctors"
)

In [0]:
#77
visits_df.createOrReplaceTempView(
    "visits"
)

In [0]:
#78
hospital_df.createOrReplaceTempView(
    "hospitals"
)


In [0]:
#79
spark.sql(
    """
    SELECT *
    FROM doctors
    """
).show()

+---------+-----------+--------------+---------+----------------+----------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|
+---------+-----------+--------------+---------+----------------+----------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|            2000|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|            1000|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|              15|            2500|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|               6|            1200|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|            3000|
|     D107| Dr. Farhan|     Neurology|     Pune|               5|            1800|
|     D108|  Dr. Nisha|   Dermatology|    Kochi|               9|            1100|
+---------+-----------+--------------+---------+----------------+----------------+



In [0]:
#80
spark.sql(
    """
    SELECT specialization,
    COUNT(*) AS total_doctors
    FROM doctors
    GROUP BY specialization
    """
).show()


+--------------+-------------+
|specialization|total_doctors|
+--------------+-------------+
|   Orthopedics|            1|
|    Cardiology|            2|
|    Pediatrics|            1|
|   Dermatology|            2|
|     Neurology|            2|
+--------------+-------------+



In [0]:
#81
spark.sql(
    """
    SELECT city,
    COUNT(*) AS total_doctors
    FROM doctors
    GROUP BY city
    """
).show()

+---------+-------------+
|     city|total_doctors|
+---------+-------------+
|    Delhi|            1|
|  Chennai|            1|
|    Kochi|            1|
|Hyderabad|            2|
|Bangalore|            1|
|     Pune|            1|
|   Mumbai|            1|
+---------+-------------+



In [0]:
#82
spark.sql(
    """
    SELECT d.doctor_name,
    SUM(v.bill_amount)
    AS revenue
    FROM doctors d
    JOIN visits v
    ON d.doctor_id=v.doctor_id
    GROUP BY d.doctor_name
    """
).show()

+-----------+-------+
|doctor_name|revenue|
+-----------+-------+
| Dr. Ramesh|10500.0|
|  Dr. Meera| 1500.0|
| Dr. Suresh|18000.0|
|  Dr. Anita| 2000.0|
|  Dr. Kiran| 7000.0|
|  Dr. Priya| 3500.0|
+-----------+-------+



In [0]:
#83
spark.sql(
    """
    SELECT d.specialization,
    SUM(v.bill_amount)
    AS revenue
    FROM doctors d
    JOIN visits v
    ON d.doctor_id=v.doctor_id
    GROUP BY d.specialization
    """
).show()

+--------------+-------+
|specialization|revenue|
+--------------+-------+
|   Orthopedics|18000.0|
|    Cardiology|17500.0|
|    Pediatrics| 1500.0|
|   Dermatology| 2000.0|
|     Neurology| 3500.0|
+--------------+-------+



In [0]:
#84
spark.sql(
    """
    SELECT d.doctor_name,
    SUM(v.bill_amount)
    AS revenue
    FROM doctors d
    JOIN visits v
    ON d.doctor_id=v.doctor_id
    GROUP BY d.doctor_name
    ORDER BY revenue DESC
    LIMIT 5
    """
).show()

+-----------+-------+
|doctor_name|revenue|
+-----------+-------+
| Dr. Suresh|18000.0|
| Dr. Ramesh|10500.0|
|  Dr. Kiran| 7000.0|
|  Dr. Priya| 3500.0|
|  Dr. Anita| 2000.0|
+-----------+-------+



In [0]:
#85
spark.sql(
    """
    SELECT *
    FROM visits
    WHERE payment_status='Pending'
    """
).show()

+--------+------------+---------+----------+------------+-----------+--------------+
|visit_id|patient_name|doctor_id|visit_date|   diagnosis|bill_amount|payment_status|
+--------+------------+---------+----------+------------+-----------+--------------+
|   V1003|  Amit Kumar|     D103|2026-01-11|Skin Allergy|     2000.0|       Pending|
|   V1008|  Meera Nair|     D103|2026-01-13|Skin Allergy|       NULL|       Pending|
+--------+------------+---------+----------+------------+-----------+--------------+



In [0]:
#86
spark.sql(
    """
    SELECT *
    FROM hospitals
    WHERE array_contains(
    services,
    'Cardiology'
    )
    """
).show()

+---------+--------------------+-----------+----------------+--------------------+
|     city|             contact|hospital_id|   hospital_name|            services|
+---------+--------------------+-----------+----------------+--------------------+
|Hyderabad|{[apollo@mail.com...|       H101| Apollo Hospital|[Cardiology, Neur...|
|Hyderabad|{[yashoda@mail.co...|       H102|Yashoda Hospital|[Cardiology, Orth...|
+---------+--------------------+-----------+----------------+--------------------+



In [0]:
#87
spark.sql(
    """
    SELECT *
    FROM hospitals
    WHERE array_contains(
    services,
    'Neurology'
    )
    """
).show()


+---------+--------------------+-----------+---------------+--------------------+
|     city|             contact|hospital_id|  hospital_name|            services|
+---------+--------------------+-----------+---------------+--------------------+
|Hyderabad|{[apollo@mail.com...|       H101|Apollo Hospital|[Cardiology, Neur...|
|Bangalore|  {NULL, 9876500013}|       H103|  Care Hospital|[Neurology, Pedia...|
+---------+--------------------+-----------+---------------+--------------------+



In [0]:
#88
spark.sql(
    """
    SELECT *
    FROM hospitals
    WHERE contact.phone IS NULL
    OR contact.email IS NULL
    """
).show()

+---------+--------------------+-----------+----------------+--------------------+
|     city|             contact|hospital_id|   hospital_name|            services|
+---------+--------------------+-----------+----------------+--------------------+
|Hyderabad|{[yashoda@mail.co...|       H102|Yashoda Hospital|[Cardiology, Orth...|
|Bangalore|  {NULL, 9876500013}|       H103|   Care Hospital|[Neurology, Pedia...|
+---------+--------------------+-----------+----------------+--------------------+



In [0]:
#89
spark.sql(
    """
    SELECT AVG(
    consultation_fee
    )
    AS average_fee
    FROM doctors
    """
).show()

+-----------+
|average_fee|
+-----------+
|     1762.5|
+-----------+



In [0]:
#90
spark.sql(
    """
    SELECT d.doctor_name,
    d.specialization,
    d.city,
    COUNT(v.visit_id)
    AS visit_count,
    SUM(v.bill_amount)
    AS total_revenue
    FROM doctors d
    JOIN visits v
    ON d.doctor_id=v.doctor_id
    GROUP BY d.doctor_name,
    d.specialization,
    d.city
    """
).show()

+-----------+--------------+---------+-----------+-------------+
|doctor_name|specialization|     city|visit_count|total_revenue|
+-----------+--------------+---------+-----------+-------------+
|  Dr. Anita|   Dermatology|  Chennai|          2|       2000.0|
|  Dr. Kiran|    Cardiology|Hyderabad|          1|       7000.0|
|  Dr. Meera|    Pediatrics|    Delhi|          1|       1500.0|
| Dr. Suresh|   Orthopedics|   Mumbai|          2|      18000.0|
| Dr. Ramesh|    Cardiology|Hyderabad|          2|      10500.0|
|  Dr. Priya|     Neurology|Bangalore|          1|       3500.0|
+-----------+--------------+---------+-----------+-------------+



In [0]:
#91
doctors_df=spark.read.csv(
    "/Workspace/Users/azuser7210_mml.local@karthikirisoutlook.onmicrosoft.com/Drafts/doctors.csv",
    header=True,
    inferSchema=True
)

visits_df=spark.read.csv(
    "/Workspace/Users/azuser7210_mml.local@karthikirisoutlook.onmicrosoft.com/Drafts/visits.csv",
    header=True,
    inferSchema=True
)

hospital_df=spark.read.option(
    "multiline",
    "true"
).json(
    "/Workspace/Users/azuser7210_mml.local@karthikirisoutlook.onmicrosoft.com/Drafts/hospital_config.json"
)

In [0]:
#92
visits_df=visits_df.fillna({
    "bill_amount":0
})

hospital_df=hospital_df.fillna({
    "contact.phone":"Not Available",
    "contact.email":"Not Available"
})

In [0]:
#93
hospital_flat=hospital_df.select(
    "hospital_id",
    "hospital_name",
    "city",
    col(
        "contact.phone"
    ).alias(
        "phone"
    ),
    col(
        "contact.email"
    ).alias(
        "email"
    ),
    explode(
        "services"
    ).alias(
        "service"
    )
)


In [0]:
#94
doctor_visit=doctors_df.join(
    visits_df,
    "doctor_id",
    "inner"
)


In [0]:
#95
doctor_visit=doctor_visit.withColumn(
    "tax",
    col(
        "bill_amount"
    )*0.05
).withColumn(
    "final_bill",
    col(
        "bill_amount"
    )+
    col(
        "tax"
    )
)


In [0]:
#96
doctor_rank=doctor_visit.groupBy(
    "doctor_name"
).agg(
    sum(
        "bill_amount"
    ).alias(
        "revenue"
    )
).withColumn(
    "rank",
    rank().over(
        Window.orderBy(
            col(
                "revenue"
            ).desc()
        )
    )
)

doctor_rank.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+-------+----+
|doctor_name|revenue|rank|
+-----------+-------+----+
| Dr. Suresh|  18000|   1|
| Dr. Ramesh|  10500|   2|
|  Dr. Kiran|   7000|   3|
|  Dr. Priya|   3500|   4|
|  Dr. Anita|   2000|   5|
|  Dr. Meera|   1500|   6|
+-----------+-------+----+



In [0]:
#97
specialization_summary=doctor_visit.groupBy(
    "specialization"
).agg(
    sum(
        "bill_amount"
    ).alias(
        "total_revenue"
    ),
    avg(
        "bill_amount"
    ).alias(
        "average_revenue"
    )
)

specialization_summary.show()

+--------------+-------------+-----------------+
|specialization|total_revenue|  average_revenue|
+--------------+-------------+-----------------+
|   Orthopedics|        18000|           9000.0|
|    Cardiology|        17500|5833.333333333333|
|    Pediatrics|         1500|           1500.0|
|   Dermatology|         2000|           1000.0|
|     Neurology|         3500|           3500.0|
+--------------+-------------+-----------------+



In [0]:
#98
doctor_visit.write.mode(
    "overwrite"
).saveAsTable(
    "silver_layer"
)

In [0]:
#99
city_report=doctor_visit.groupBy(
    "city"
).sum(
    "bill_amount"
)

specialization_report=doctor_visit.groupBy(
    "specialization"
).sum(
    "bill_amount"
)

city_report.show()
specialization_report.show()

+---------+----------------+
|     city|sum(bill_amount)|
+---------+----------------+
|    Delhi|            1500|
|  Chennai|            2000|
|Hyderabad|           17500|
|Bangalore|            3500|
|   Mumbai|           18000|
+---------+----------------+

+--------------+----------------+
|specialization|sum(bill_amount)|
+--------------+----------------+
|   Orthopedics|           18000|
|    Cardiology|           17500|
|    Pediatrics|            1500|
|   Dermatology|            2000|
|     Neurology|            3500|
+--------------+----------------+



In [0]:
#100
dashboard_df=doctor_visit.groupBy(
    "doctor_name",
    "specialization",
    "city"
).agg(
    count(
        "visit_id"
    ).alias(
        "total_visits"
    ),
    sum(
        "bill_amount"
    ).alias(
        "total_revenue"
    ),
    avg(
        "bill_amount"
    ).alias(
        "average_bill"
    )
)

dashboard_df.show()

+-----------+--------------+---------+------------+-------------+------------+
|doctor_name|specialization|     city|total_visits|total_revenue|average_bill|
+-----------+--------------+---------+------------+-------------+------------+
|  Dr. Anita|   Dermatology|  Chennai|           2|         2000|      1000.0|
|  Dr. Kiran|    Cardiology|Hyderabad|           1|         7000|      7000.0|
|  Dr. Meera|    Pediatrics|    Delhi|           1|         1500|      1500.0|
| Dr. Suresh|   Orthopedics|   Mumbai|           2|        18000|      9000.0|
| Dr. Ramesh|    Cardiology|Hyderabad|           2|        10500|      5250.0|
|  Dr. Priya|     Neurology|Bangalore|           1|         3500|      3500.0|
+-----------+--------------+---------+------------+-------------+------------+



In [0]:
#89
spark.sql(
    """
    SELECT AVG(
    consultation_fee
    )
    AS average_fee
    FROM doctors
    """
).show()

+-----------+
|average_fee|
+-----------+
|     1762.5|
+-----------+

